In [1]:
# optuna_train.py
import torch
import torch.nn.functional as F
from torch_geometric.datasets import QM9
from torch_geometric.loader import DataLoader
import optuna
from sklearn.model_selection import train_test_split
from model import WDMPNNModel  # 你的模型文件
from train import WMAELoss, compute_task_stats, run_training  # 你写好的train工具
import numpy as np
import pandas as pd

d:\anaconda\envs\neurips\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def suggest_params(trial):
    return {
        # === Encoder ===
        "hidden_dim": trial.suggest_categorical("hidden_dim", [128, 256, 512]),
        "num_layers": trial.suggest_int("num_layers", 2, 6),
        "act": trial.suggest_categorical("act", ["relu", "silu", "mish"]),
        "dropout": trial.suggest_float("dropout", 0.0, 0.5),

        # === Attention ===
        "use_edge_attn": trial.suggest_categorical("use_edge_attn", [True, False]),
        "att_hidden": trial.suggest_int("att_hidden", 32, 128),

        # === Pooling ===
        "pool": trial.suggest_categorical("pool", ["mean", "att"]),

        # === Adapter ===
        "adapter_kind": trial.suggest_categorical("adapter_kind", ["none", "linear", "mlp"]),
        "adapter_hidden": trial.suggest_int("adapter_hidden", 16, 128),
        "adapter_dropout": trial.suggest_float("adapter_dropout", 0.0, 0.3),

        # === Head ===
        "mlp_hidden": trial.suggest_categorical("mlp_hidden", [
            (128, 64),
            (256, 128),
            (256, 128, 64),
        ]),
        "head_dropout": trial.suggest_float("head_dropout", 0.0, 0.5),

        # === Optimizer ===
        "lr_encoder": trial.suggest_loguniform("lr_encoder", 1e-5, 5e-4),
        "lr_adapter": trial.suggest_loguniform("lr_adapter", 1e-4, 1e-3),
        "lr_head": trial.suggest_loguniform("lr_head", 1e-4, 1e-3),
        "weight_decay": trial.suggest_loguniform("weight_decay", 1e-6, 1e-2),

        # === Training ===
        "batch_size": trial.suggest_categorical("batch_size", [32, 64, 128]),
    }

In [ ]:

device = "cuda" if torch.cuda.is_available() else "cpu"


# -------------------- 数据加载 --------------------
def load_qm9(batch_size=64, num_workers=0):
    dataset = QM9(root="kaggle/working/qm9")
    # QM9 有 19 个 regression targets，存储在 data.y 中
    idx = list(range(len(dataset)))
    train_idx, val_idx = train_test_split(idx, test_size=0.1, random_state=42)

    train_ds = dataset[train_idx]
    val_ds = dataset[val_idx]

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    # 转换成 DataFrame 方便统计 n_dict / r_dict
    y = dataset.data.y.numpy()
    columns = [f"task_{i}" for i in range(y.shape[1])]
    df = pd.DataFrame(y, columns=columns)

    return train_loader, val_loader, df, columns, dataset


# -------------------- Optuna 目标函数 --------------------
def objective(trial):
    # 超参数搜索空间
    hidden_dim = trial.suggest_categorical("hidden_dim", [64, 128, 256])
    num_layers = trial.suggest_int("num_layers", 2, 6)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    act = trial.suggest_categorical("act", ["relu", "silu", "mish"])
    pool = trial.suggest_categorical("pool", ["mean", "att"])
    use_edge_attn = trial.suggest_categorical("use_edge_attn", [False, True])
    lr = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    mlp_hidden = trial.suggest_categorical(
        "mlp_hidden",
        [(128, 64), (256, 128), (256, 128, 64)]  # 改成 tuple
    )
    mlp_hidden = list(mlp_hidden)  # 转回 list 给模型用

    # 数据加载
    train_loader, val_loader, df, tasks, dataset = load_qm9(batch_size=64)
    node_dim = dataset.num_node_features
    edge_dim = dataset.num_edge_features

    # 构建模型
    model = WDMPNNModel(
        node_dim=node_dim,
        edge_dim=edge_dim,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        tasks=tasks,
        mlp_hidden=mlp_hidden,
        use_edge_attn=use_edge_attn,
        dropout=dropout,
        act=act,
        pool=pool,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # 跑训练
    model = run_training(
        model,
        train_loader,
        val_loader,
        optimizer,
        tasks,
        df,
        device=device,
        max_epochs=30,   # 预训练时可以跑久一些，比如 100；这里快速调参
        patience=10,
    )

    # 在验证集上评估
    n_dict, r_dict = compute_task_stats(df, tasks)
    loss_fn = WMAELoss(tasks, n_dict, r_dict)
    val_loss, _ = evaluate(model, val_loader, loss_fn, device, tasks)

    return val_loss

In [8]:
train_loader, val_loader, df, tasks, dataset= load_qm9(batch_size=64)
node_dim = dataset.num_node_features
edge_dim = dataset.num_edge_features

In [9]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)  # 跑30个实验，可以加大

print("Best trial:", study.best_trial.params)

[I 2025-09-10 08:13:47,815] A new study created in memory with name: no-name-52d86c3c-18ef-4923-b52a-1e2679aa4b52
d:\anaconda\envs\neurips\lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [128, 64] which is of type list.
  warnings.warn(message)
d:\anaconda\envs\neurips\lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [256, 128] which is of type list.
  warnings.warn(message)
d:\anaconda\envs\neurips\lib\site-packages\optuna\distributions.py:518: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains [256, 128, 64] which is of type list.
  warnings.warn(message)
[W 2025-09-10 08:16:14,133] Trial 0 failed with parameters: {'hidden_d

KeyboardInterrupt: 